# Retrieval-Augmented Generation (RAG) for Tesla Annual Reports

## Project Overview

This project demonstrates the implementation of a Retrieval-Augmented Generation (RAG) system for answering questions from Tesla's annual reports (2019–2023).

The solution combines:

- Vector embeddings using Sentence Transformers
- ChromaDB for vector storage and retrieval
- Llama 3.1 70B for answer generation
- Advanced retrieval using Hypothetical Question Indexing

The objective is to improve factual accuracy and reduce hallucinations by grounding responses in company filings.

---

## Business Use Case

Financial analysts often need to extract information from hundreds of pages of annual reports. Traditional keyword search is limited because users rarely phrase questions exactly as information appears in documents.

This RAG system enables:

- Natural language querying
- Semantic document retrieval
- Grounded financial analysis
- Explainable AI responses

---

## Architecture

User Question
↓
Embedding Model
↓
Vector Search (ChromaDB)
↓
Relevant Document Chunks
↓
LLM (Llama 3.1 70B)
↓
Final Answer


# Dataset Description

The dataset consists of Tesla Annual Reports (Form 10-K) spanning 2019–2023.

These reports contain:

- Revenue information
- Automotive sales performance
- Energy generation and storage metrics
- Operating expenses
- Risk factors
- Corporate governance details

The reports collectively contain several hundred pages of financial disclosures.


# Environment Setup

The following libraries are required:

- ChromaDB
- LangChain
- Sentence Transformers
- NVIDIA API SDK
- PyPDF Loader

The environment is configured before initializing the retrieval pipeline.

In [3]:
import time
import chromadb
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader

from datetime import datetime


In [4]:
from dotenv import load_dotenv
load_dotenv()

import os
os.environ["NVIDIA_API_KEY"] = os.getenv('NVIDIA_API_KEY')

In [5]:
# from groq import Groq

# client = Groq()

from openai import OpenAI
import os

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv('NVIDIA_API_KEY')
)

In [34]:
model_name = "meta/llama-3.1-70b-instruct"
# model_name = "meta/llama-3.1-8b-instruct"

 Instantiating the embedding model

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Ashish Sinha\AppData\Local\Temp\ipykernel_12428\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


# Load the Vector Database

Since we persisted the database to to a folder, we can upload this database to this Colab instance and point a Chroma instance to this database.

In practise, the database is maintained as a separate entity and CRUD operations are managed just as one would for normal databases (e.g., relational databases).

Now that the database is uploaded onto the Colab instance, we can unzip it and attach a retriever.

In [8]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [10]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000023912E12270>, search_kwargs={'k': 5})

# RAG Q&A

## Prompt Design

The RAG system message should clearly communicate to the LLM that the input will include a user query along with the necessary context information to address that query. Additionally, the response should rely solely on the context information provided.

In [11]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [12]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

## Retrieving relevant documents

In [13]:
user_query = "What was the automotive revenue in 2021?"

In [14]:
relevant_document_chunks = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [15]:
len(relevant_document_chunks)

5

We can inspect the first document like so:

In [16]:
for document in relevant_document_chunks:
    print(document)
    break

page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

In [17]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
2022
2021
$
%
$
%
Automotive sales
$
78,509 
$
67,210 
$
44,125 
$
11,299 
17 
%
$
23,085 
52 
%
Automotive regulatory credits
1,790 
1,776 
1,465 
14 
1 
%
311 
21 
%
Automotive leasing
2,120 
2,476 
1,642 
(356)
(14)
%
834 
51 
%
Total automotive revenues
82,419 
71,462 
47,232 
10,957 
15 
%
24,230 
51 
%
Services and other
8,319 
6,091 
3,802 
2,228 
37 
%
2,289 
60 
%
Total automotive & services and other segment
revenue
90,738 
77,553 
51,034 
13,185 
17 
%
26,519 
52 
%
Energy generation and storage segment revenue
6,035 
3,909 
2,789 
2,126 
54 
%
1,120 
40 
%
Total revenues
$
96,773 
$
81,462 
$
53,823 
$
15,311 
19 
%
$
27,639 
51 
%
Automotive & Services and Other Segment
Automotive sales revenue includes revenues related to cash and financing deliveries of new Model S, Model X, Semi, Model 3, Model Y, and
Cybertruck vehicles, including access to our FSD Capability features and their ongoing maintenance, internet connectivity, free Supercharging program

## Composing the response

To compose the response to user queries, we assemble the prompt that uses the system message defined above and the dynamically retrieved context for the user query.

In [18]:
user_query = "What was the automotive revenue in 2021?"

In [19]:

# model_name = 'openai/gpt-oss-120b'
model_name = "meta/llama-3.1-70b-instruct"

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

$71,462


# Advanced Retrieval Strategy: Hypothetical Question Indexing

## Motivation

Standard semantic search may fail when a user query differs significantly from the wording used in the source documents.

To improve retrieval quality, we generate synthetic questions for every document chunk.

Instead of embedding only document text, we embed:

- Document chunks
- Generated hypothetical questions

This improves recall and enables more robust semantic matching.

# Hypothetical Questions

In this approach, we generate 3 hypothetical questions that can be answered with each document chunk. These hypothetical questions are then seperately indexed into the vector database (along with the parent document chunk ids as metadata). For each query, we then retrieve relevant hypothetical questions first and the then retrieve the associated chunks as the second step. Note that the retrieval is focused on the comparison between the user query and hypothetical questions.

# Lets Create Database for Hypothetical Questions

In [ ]:
# pdf_folder_location = "tesla-annual-reports"

In [ ]:
# pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [ ]:
# type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [ ]:
# text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     encoding_name='cl100k_base',
#     chunk_size=512,
#     chunk_overlap=16
# )

In [ ]:
# collection = vectorstore_persisted._collection
# total_docs = collection.count()
# print(total_docs)

3337


In [ ]:
# tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

Generating Hypothetical Questions: 0it [01:38, ?it/s]


In [20]:
import uuid
import os
import re

pdf_folder_location = "tesla-annual-reports"

pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=512,
    chunk_overlap=16
)

tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

# Add Assignment Metadata

for chunk in tesla_10k_chunks:

    source_path = chunk.metadata.get("source", "")

    source_doc = os.path.basename(source_path)

    year_match = re.search(r"(2019|2020|2021|2022|2023)", source_doc)

    year = int(year_match.group(1)) if year_match else None

    chunk.metadata.update({
        "chunk_id": str(uuid.uuid4()),
        "source_doc": source_doc,
        "year": year,
        "section": "Unknown"  # can improve later
    })

print(f"Total chunks created: {len(tesla_10k_chunks)}")

Total chunks created: 3337


In [21]:
tesla_10k_chunks[0].metadata

{'producer': 'Qt 5.11.3',
 'creator': 'wkhtmltopdf 0.12.5',
 'creationdate': '2022-05-02T10:10:26+00:00',
 'title': '',
 'source': 'tesla-annual-reports\\tsla-10ka_20211231-gen.pdf',
 'total_pages': 56,
 'page': 0,
 'page_label': '1',
 'chunk_id': '34ead2d5-bcbd-482f-8104-46662b3cd246',
 'source_doc': 'tsla-10ka_20211231-gen.pdf',
 'year': 2021,
 'section': 'Unknown'}

# Database Creation

In [22]:
NEW_COLLECTION = "tesla-10k-2019-to-2023-v2"

In [25]:
from langchain_chroma import Chroma

NEW_COLLECTION = "tesla-10k-2019-to-2023-v2"

vectorstore = Chroma.from_documents(
    documents=tesla_10k_chunks,
    embedding=embedding,
    collection_name=NEW_COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
    client=chromadb_client,
    persist_directory="./tesla_db"
)

print("Collection created successfully")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Collection created successfully


In [26]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [27]:
docs = retriever.invoke(
    "What was Tesla automotive revenue in 2021?"
)

docs[0].metadata

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


{'chunk_id': '645a79f9-b568-4200-8c1f-8616170c396d',
 'creationdate': '2023-01-31T11:10:39+00:00',
 'creator': 'wkhtmltopdf 0.12.6',
 'page': 87,
 'page_label': '88',
 'producer': 'Qt 5.15.2',
 'section': 'Unknown',
 'source': 'tesla-annual-reports\\tsla-20221231-gen.pdf',
 'source_doc': 'tsla-20221231-gen.pdf',
 'title': '',
 'total_pages': 251,
 'year': 2022}

In [28]:
chromadb_client.list_collections()

['tesla-10k-2019-to-2023-v2',
 'tesla-10k-2019-to-2023',
 'hypothetical_questions']

In [ ]:
# i = 0 # Initialize the starting index for the chunks

# while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
#     vectorstore.add_documents( # Add documents to the vector store in batches of 1500
#         documents=tesla_10k_chunks[i:i+500], # Get the current batch of 1500 chunks
#         ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
#     )

#     i += 500 # Increment the index by 500 to move to the next batch
#     time.sleep(5) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [29]:
hypothetical_questions_system_message = """
You are generating hypothetical retrieval questions for a RAG system built on Tesla 10-K filings.

Generate EXACTLY 3 questions that can be answered using ONLY the provided document chunk.

Requirements:

- Questions must be specific and useful for semantic retrieval.
- Include a mix of:
  * factual questions
  * analytical questions
  * comparative questions (when applicable)
  * risk-oriented questions (when applicable)
- Do not introduce facts, entities, dates, numbers, or events not present in the chunk.
- Questions should help retrieve the chunk even when users ask business-oriented or indirect questions.
- Each question must be on a separate line.
- Do not number questions.
- Do not use bullet points.
- Do not include explanations.
- Return only the questions.
"""

user_message_template = """
<Document>
{document}
</Document>
"""

In [179]:
# response = client.chat.completions.create(
#     model=model_name,
#     temperature=0.2,
#     max_tokens=300,
#     messages=[
#         {
#             "role": "system",
#             "content": hypothetical_questions_system_message
#         },
#         {
#             "role": "user",
#             "content": user_message_template.format(
#                 document=chunk.page_content
#             )
#         }
#     ]
# )

In [180]:
# Document(
#     page_content=question,
#     metadata={
#         "parent_chunk_id": chunk.metadata["chunk_id"],
#         "source_doc": chunk.metadata["source_doc"],
#         "year": chunk.metadata["year"],
#         "section": chunk.metadata["section"]
#     }
# )

In [30]:
len(tesla_10k_chunks)

3337

In [184]:
# from tqdm.auto import tqdm
# from langchain.schema import Document
# import json
# import time

# hypothetical_questions = []
# questions_json = []

# batch_size = 10

# successful_batches = 0
# failed_batches = 0
# skipped_batches = 0

# # Adjust according to model context window
# MAX_CHARS = 100000

# total_batches = (
#     len(tesla_10k_chunks) + batch_size - 1
# ) // batch_size

# for batch_num, i in enumerate(
#     tqdm(
#         range(0, len(tesla_10k_chunks), batch_size),
#         total=total_batches,
#         desc="Generating Questions"
#     ),
#     start=1
# ):

#     batch_docs = tesla_10k_chunks[i:i + batch_size]

#     combined_text = "\n\n".join(
#         doc.page_content
#         for doc in batch_docs
#     )

#     char_count = len(combined_text)

#     print(
#         f"\nBatch {batch_num}/{total_batches}"
#         f" | Chars={char_count:,}"
#         f" | Docs={len(batch_docs)}"
#     )

#     # Skip oversized batches
#     if char_count > MAX_CHARS:

#         skipped_batches += 1

#         print(
#             f"Skipping batch {batch_num}"
#             f" (too large: {char_count:,} chars)"
#         )

#         continue

#     questions = ""

#     wait_times = [5, 10, 15]

#     for retry in range(len(wait_times)):

#         try:

#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": user_message_template.format(
#                             document=combined_text
#                         )
#                     }
#                 ],
#                 temperature=0
#             )

#             questions = (
#                 response
#                 .choices[0]
#                 .message
#                 .content
#                 .strip()
#             )

#             successful_batches += 1

#             print(
#                 f" Success"
#                 f" | Successful Batches: {successful_batches}"
#             )

#             break

#         except Exception as e:

#             error_msg = str(e).lower()

#             print(
#                 f"\n Batch {batch_num}"
#                 f" | Retry {retry + 1}"
#             )
#             print(e)

#             # Last retry
#             if retry == len(wait_times) - 1:

#                 failed_batches += 1

#                 print(
#                     f"Failed after "
#                     f"{len(wait_times)} attempts"
#                 )

#                 questions = ""

#                 break

#             wait_time = wait_times[retry]

#             print(
#                 f"Waiting {wait_time}s "
#                 f"before retry..."
#             )

#             time.sleep(wait_time)

#     if not questions:
#         continue

#     questions_list = [
#         q.strip()
#         for q in questions.split("\n")
#         if q.strip()
#     ]

#     parent_doc = batch_docs[0]

#     chunk_id = (
#         parent_doc.metadata.get("chunk_id")
#         or f"chunk_{i}"
#     )

#     for question in questions_list:

#         metadata = {
#             "parent_chunk_id": str(chunk_id),
#             "parent_collection": "tesla_10k_collection"
#         }

#         # For Chroma
#         hypothetical_questions.append(
#             Document(
#                 page_content=question,
#                 metadata=metadata
#             )
#         )

#         # For JSON
#         questions_json.append({
#             "question": question,
#             "parent_chunk_id": str(chunk_id),
#             "parent_collection": "tesla_10k_collection"
#         })

#     # Small delay between successful requests
#     time.sleep(1)

# print("\n" + "=" * 60)
# print("SUMMARY")
# print("=" * 60)

# print(f"Total Batches      : {total_batches}")
# print(f"Successful Batches : {successful_batches}")
# print(f"Failed Batches     : {failed_batches}")
# print(f"Skipped Batches    : {skipped_batches}")
# print(f"Questions Created  : {len(hypothetical_questions)}")

# # Save JSON
# with open(
#     "tesla_hypothetical_questions.json",
#     "w",
#     encoding="utf-8"
# ) as f:

#     json.dump(
#         questions_json,
#         f,
#         indent=4,
#         ensure_ascii=False
#     )

# print("\nJSON saved successfully.")

In [31]:
# from tqdm.auto import tqdm
# from langchain.schema import Document
# import json
# import time

# hypothetical_questions = []
# questions_json = []

# batch_size = 25

# successful_batches = 0
# failed_batches = 0

# total_batches = (
#     len(tesla_10k_chunks) + batch_size - 1
# ) // batch_size

# for batch_num, i in enumerate(
#     tqdm(
#         range(0, len(tesla_10k_chunks), batch_size),
#         total=total_batches,
#         desc="Generating HQ Questions"
#     ),
#     start=1
# ):

#     batch_docs = tesla_10k_chunks[i:i + batch_size]

#     formatted_chunks = []

#     for idx, doc in enumerate(batch_docs):

#         formatted_chunks.append(
#             f"""
# CHUNK_INDEX: {idx}

# {doc.page_content[:2000]}
# """
#         )

#     batch_text = "\n\n".join(formatted_chunks)

#     wait_times = [5, 10, 20]

#     response_text = ""

#     for retry in range(len(wait_times)):

#         try:

#             response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": hypothetical_questions_system_message
#                     },
#                     {
#                         "role": "user",
#                         "content": f"""
# For each chunk below generate EXACTLY 3 questions.

# Return VALID JSON ONLY.

# Format:

# [
#   {{
#     "chunk_index": 0,
#     "questions": [
#       "...",
#       "...",
#       "..."
#     ]
#   }}
# ]

# {batch_text}
# """
#                     }
#                 ],
#                 temperature=0.2,
#                 max_tokens=4000
#             )

#             response_text = (
#                 response
#                 .choices[0]
#                 .message
#                 .content
#                 .strip()
#             )

#             successful_batches += 1
#             break

#         except Exception as e:

#             print(
#                 f"\nBatch {batch_num}"
#                 f" | Retry {retry + 1}"
#             )

#             print(e)

#             if retry == len(wait_times) - 1:

#                 failed_batches += 1
#                 response_text = ""
#                 break

#             time.sleep(wait_times[retry])

#     if not response_text:
#         continue

#     try:

#         # Remove markdown fences if model adds them
#         response_text = response_text.replace(
#             "```json", ""
#         ).replace(
#             "```", ""
#         ).strip()

#         generated_data = json.loads(response_text)

#     except Exception as e:

#         print(
#             f"\nJSON Parse Failed "
#             f"for batch {batch_num}"
#         )

#         print(e)

#         failed_batches += 1
#         continue

#     for item in generated_data:

#         chunk_idx = item["chunk_index"]

#         if chunk_idx >= len(batch_docs):
#             continue

#         parent_chunk = batch_docs[chunk_idx]

#         for question in item["questions"][:3]:

#             metadata = {
#                 "parent_chunk_id":
#                     parent_chunk.metadata["chunk_id"],
#                 "source_doc":
#                     parent_chunk.metadata["source_doc"],
#                 "year":
#                     parent_chunk.metadata["year"],
#                 "section":
#                     parent_chunk.metadata["section"]
#             }

#             hypothetical_questions.append(
#                 Document(
#                     page_content=question,
#                     metadata=metadata
#                 )
#             )

#             questions_json.append({
#                 "hypothetical_question": question,
#                 "parent_chunk_id":
#                     parent_chunk.metadata["chunk_id"],
#                 "source_doc":
#                     parent_chunk.metadata["source_doc"],
#                 "year":
#                     parent_chunk.metadata["year"],
#                 "section":
#                     parent_chunk.metadata["section"]
#             })

#     # small pause to reduce 429s
#     time.sleep(2)

# print("\n" + "=" * 60)
# print("SUMMARY")
# print("=" * 60)

# print(f"Successful Batches : {successful_batches}")
# print(f"Failed Batches     : {failed_batches}")
# print(
#     f"Questions Created  : "
#     f"{len(hypothetical_questions)}"
# )

# with open(
#     "tesla_hypothetical_questions.json",
#     "w",
#     encoding="utf-8"
# ) as f:

#     json.dump(
#         questions_json,
#         f,
#         indent=4,
#         ensure_ascii=False
#     )

# print("\nJSON saved successfully.")

In [35]:
from tqdm.auto import tqdm
from langchain.schema import Document
from datetime import datetime
import json
import time

# ==========================================================
# CONFIG
# ==========================================================

batch_size = 25

# Optional for testing
source_chunks = tesla_10k_chunks[:500]

# source_chunks = tesla_10k_chunks

# ==========================================================
# STORAGE
# ==========================================================

hypothetical_questions = []
questions_json = []

successful_batches = 0
failed_batches = 0

# ==========================================================
# TIMING
# ==========================================================

start_time = time.time()

total_batches = (
    len(source_chunks) + batch_size - 1
) // batch_size

print("=" * 100)
print("TESLA HQ GENERATION STARTED")
print("=" * 100)

print(
    f"Start Time      : "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

print(
    f"Total Chunks    : "
    f"{len(source_chunks):,}"
)

print(
    f"Batch Size      : "
    f"{batch_size}"
)

print(
    f"Total Batches   : "
    f"{total_batches}"
)

print("=" * 100)

# ==========================================================
# MAIN LOOP
# ==========================================================

for batch_num, i in enumerate(
    tqdm(
        range(
            0,
            len(source_chunks),
            batch_size
        ),
        total=total_batches,
        desc="Generating HQ Questions"
    ),
    start=1
):

    batch_start_time = time.time()

    batch_docs = source_chunks[
        i:i + batch_size
    ]

    # ------------------------------------------------------
    # BUILD BATCH INPUT
    # ------------------------------------------------------

    formatted_chunks = []

    for idx, doc in enumerate(batch_docs):

        formatted_chunks.append(
            f"""
CHUNK_INDEX: {idx}

{doc.page_content[:1000]}
"""
        )

    batch_text = "\n\n".join(
        formatted_chunks
    )

    response_text = ""

    wait_times = [5, 10, 20]

    # ------------------------------------------------------
    # API CALL
    # ------------------------------------------------------

    for retry in range(
        len(wait_times)
    ):

        try:

            response = (
                client.chat.completions.create(
                    model=model_name,
                    messages=[
                        {
                            "role": "system",
                            "content":
                                hypothetical_questions_system_message
                        },
                        {
                            "role": "user",
                            "content": f"""
For EACH chunk below:

1. Generate EXACTLY 3 hypothetical questions.
2. Questions must be answerable using ONLY that chunk.
3. Do not introduce facts not present in the chunk.
4. Return VALID JSON ONLY.

Required Format:

[
  {{
    "chunk_index": 0,
    "questions": [
      "...",
      "...",
      "..."
    ]
  }}
]

{batch_text}
"""
                        }
                    ],
                    temperature=0.2,
                    max_tokens=3000
                )
            )

            response_text = (
                response
                .choices[0]
                .message
                .content
                .strip()
            )

            successful_batches += 1

            break

        except Exception as e:

            print(
                f"\nBatch {batch_num}"
                f" | Retry {retry + 1}"
            )

            print(e)

            if retry == (
                len(wait_times) - 1
            ):

                failed_batches += 1
                response_text = ""

                break

            time.sleep(
                wait_times[retry]
            )

    # ------------------------------------------------------
    # SKIP FAILED
    # ------------------------------------------------------

    if not response_text:
        continue

    # ------------------------------------------------------
    # PARSE JSON
    # ------------------------------------------------------

    try:

        response_text = (
            response_text
            .replace(
                "```json",
                ""
            )
            .replace(
                "```",
                ""
            )
            .strip()
        )

        generated_data = json.loads(
            response_text
        )

    except Exception as e:

        print(
            f"\nJSON Parse Failed "
            f"for Batch {batch_num}"
        )

        print(e)

        failed_batches += 1

        continue

    # ------------------------------------------------------
    # SAVE QUESTIONS
    # ------------------------------------------------------

    batch_question_count = 0

    for item in generated_data:

        chunk_idx = item.get(
            "chunk_index"
        )

        if (
            chunk_idx is None
            or chunk_idx >= len(batch_docs)
        ):
            continue

        parent_chunk = (
            batch_docs[chunk_idx]
        )

        questions = item.get(
            "questions",
            []
        )[:3]

        for question in questions:

            metadata = {
                "parent_chunk_id":
                    parent_chunk.metadata[
                        "chunk_id"
                    ],
                "source_doc":
                    parent_chunk.metadata[
                        "source_doc"
                    ],
                "year":
                    parent_chunk.metadata[
                        "year"
                    ],
                "section":
                    parent_chunk.metadata[
                        "section"
                    ]
            }

            hypothetical_questions.append(
                Document(
                    page_content=question,
                    metadata=metadata
                )
            )

            questions_json.append(
                {
                    "hypothetical_question":
                        question,
                    "parent_chunk_id":
                        parent_chunk.metadata[
                            "chunk_id"
                        ],
                    "source_doc":
                        parent_chunk.metadata[
                            "source_doc"
                        ],
                    "year":
                        parent_chunk.metadata[
                            "year"
                        ],
                    "section":
                        parent_chunk.metadata[
                            "section"
                        ]
                }
            )

            batch_question_count += 1

    # ------------------------------------------------------
    # CHECKPOINT
    # ------------------------------------------------------

    if batch_num % 10 == 0:

        checkpoint_file = (
            "tesla_hq_checkpoint.json"
        )

        with open(
            checkpoint_file,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                questions_json,
                f,
                indent=2,
                ensure_ascii=False
            )

        print(
            f"\nCheckpoint Saved:"
            f" {checkpoint_file}"
        )

    # ------------------------------------------------------
    # STATS
    # ------------------------------------------------------

    elapsed_time = (
        time.time() - start_time
    )

    avg_batch_time = (
        elapsed_time / batch_num
    )

    remaining_batches = (
        total_batches - batch_num
    )

    eta_seconds = (
        avg_batch_time *
        remaining_batches
    )

    batch_runtime = (
        time.time() -
        batch_start_time
    )

    print("\n" + "-" * 100)

    print(
        f"Batch "
        f"{batch_num}/{total_batches}"
        f" Complete"
    )

    print(
        f"Questions This Batch : "
        f"{batch_question_count}"
    )

    print(
        f"Questions Total      : "
        f"{len(questions_json):,}"
    )

    print(
        f"Successful Batches   : "
        f"{successful_batches}"
    )

    print(
        f"Failed Batches       : "
        f"{failed_batches}"
    )

    print(
        f"Batch Time           : "
        f"{batch_runtime:.1f}s"
    )

    print(
        f"Average Batch Time   : "
        f"{avg_batch_time:.1f}s"
    )

    print(
        f"Elapsed Time         : "
        f"{elapsed_time/60:.1f} min"
    )

    print(
        f"ETA Remaining        : "
        f"{eta_seconds/60:.1f} min"
    )

    print("-" * 100)

    # Reduce 429 errors
    time.sleep(2)

# ==========================================================
# FINAL SAVE
# ==========================================================

with open(
    "tesla_hypothetical_questions.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        questions_json,
        f,
        indent=4,
        ensure_ascii=False
    )

# ==========================================================
# SUMMARY
# ==========================================================

total_runtime = (
    time.time() -
    start_time
)

print("\n" + "=" * 100)
print("HQ GENERATION COMPLETE")
print("=" * 100)

print(
    f"Runtime              : "
    f"{total_runtime/60:.2f} minutes"
)

print(
    f"Successful Batches   : "
    f"{successful_batches}"
)

print(
    f"Failed Batches       : "
    f"{failed_batches}"
)

print(
    f"Questions Generated  : "
    f"{len(hypothetical_questions):,}"
)

print(
    f"JSON Records         : "
    f"{len(questions_json):,}"
)

print(
    f"Completed At         : "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

print("=" * 100)

TESLA HQ GENERATION STARTED
Start Time      : 2026-06-04 15:48:31
Total Chunks    : 500
Batch Size      : 25
Total Batches   : 20


Generating HQ Questions:   0%|          | 0/20 [00:00<?, ?it/s]


----------------------------------------------------------------------------------------------------
Batch 1/20 Complete
Questions This Batch : 75
Questions Total      : 75
Successful Batches   : 1
Failed Batches       : 0
Batch Time           : 580.9s
Average Batch Time   : 580.9s
Elapsed Time         : 9.7 min
ETA Remaining        : 184.0 min
----------------------------------------------------------------------------------------------------


Generating HQ Questions:   5%|▌         | 1/20 [09:42<3:04:35, 582.90s/it]


----------------------------------------------------------------------------------------------------
Batch 2/20 Complete
Questions This Batch : 75
Questions Total      : 150
Successful Batches   : 2
Failed Batches       : 0
Batch Time           : 572.3s
Average Batch Time   : 577.6s
Elapsed Time         : 19.3 min
ETA Remaining        : 173.3 min
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
Batch 19/20 Complete
Questions This Batch : 75
Questions Total      : 1,347
Successful Batches   : 19
Failed Batches       : 0
Batch Time           : 143.9s
Average Batch Time   : 406.5s
Elapsed Time         : 128.7 min
ETA Remaining        : 6.8 min
----------------------------------------------------------------------------------------------------


Generating HQ Questions:  95%|█████████▌| 19/20 [2:08:44<08:30, 510.35s/it]


Checkpoint Saved: tesla_hq_checkpoint.json

----------------------------------------------------------------------------------------------------
Batch 20/20 Complete
Questions This Batch : 75
Questions Total      : 1,422
Successful Batches   : 20
Failed Batches       : 0
Batch Time           : 487.3s
Average Batch Time   : 410.6s
Elapsed Time         : 136.9 min
ETA Remaining        : 0.0 min
----------------------------------------------------------------------------------------------------


Generating HQ Questions: 100%|██████████| 20/20 [2:16:53<00:00, 410.69s/it]


HQ GENERATION COMPLETE
Runtime              : 136.90 minutes
Successful Batches   : 20
Failed Batches       : 0
Questions Generated  : 1,422
JSON Records         : 1,422
Completed At         : 2026-06-04 18:05:25


In [36]:
print(len(hypothetical_questions))

1422


In [38]:
questions_json = []

In [39]:
questions_json[:5]

[]

### Create HQ Collection

In [40]:
hq_collection_name = "tesla-hypothetical-questions"

hq_vectorstore = Chroma(
    collection_name=hq_collection_name,
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db",
    collection_metadata={"hnsw:space": "cosine"}
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Create HQ Retriever

In [41]:
hq_retriever = hq_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

hq_retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002394015B2F0>, search_kwargs={'k': 5})

### Test HQ Retrieval

In [42]:
query = """
What should a board member ask about risks
that could prevent Tesla from meeting
production, delivery, or scaling expectations?
"""

In [59]:
chunk_lookup = {
    doc.metadata["chunk_id"]: doc
    for doc in tesla_10k_chunks
}

In [ ]:
hq_results = hq_retriever.invoke(query)

parent_chunks = [] 

for doc in hq_results:

    parent_id = doc.metadata["parent_chunk_id"]

    if parent_id in chunk_lookup:

        parent_chunks.append(
            chunk_lookup[parent_id]
        )

In [61]:
parent_chunks = []

for doc in hq_results:

    parent_id = doc.metadata[
        "parent_chunk_id"
    ]

    if parent_id in chunk_lookup:

        parent_chunks.append(
            chunk_lookup[parent_id]
        )

# Benchmark Questions

In [62]:
benchmark_questions = {
    "HQ1": "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",

    "HQ2": "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",

    "HQ3": "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",

    "HQ4": "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
}

### Baseline Retrieval Function

In [63]:
def retrieve_baseline(query, k=5):

    docs = vectorstore_persisted.similarity_search(
        query,
        k=k
    )

    return docs

### HQ Retrieval Function

In [64]:
def retrieve_hq(query, k=5):

    hq_results = (
        hq_vectorstore
        .similarity_search_with_score(
            query,
            k=k
        )
    )

    retrieved_questions = []

    parent_chunks = []

    seen_chunks = set()

    for doc, score in hq_results:

        retrieved_questions.append({
            "hypothetical_question":
                doc.page_content,
            "parent_chunk_id":
                doc.metadata["parent_chunk_id"],
            "source_doc":
                doc.metadata["source_doc"],
            "year":
                doc.metadata["year"],
            "section":
                doc.metadata["section"],
            "score":
                float(score)
        })

        parent_id = (
            doc.metadata[
                "parent_chunk_id"
            ]
        )

        if (
            parent_id in chunk_lookup
            and parent_id not in seen_chunks
        ):

            parent_chunks.append(
                chunk_lookup[parent_id]
            )

            seen_chunks.add(
                parent_id
            )

    return (
        retrieved_questions,
        parent_chunks
    )

### Answer Generation

In [65]:
def generate_answer(
    query,
    parent_chunks
):

    context = "\n\n".join(
        chunk.page_content
        for chunk in parent_chunks
    )

    response = (
        client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content":
                    """
                    Answer ONLY using the provided context.

                    If evidence is insufficient,
                    explicitly say so.

                    Cite supporting sections
                    whenever possible.
                    """
                },
                {
                    "role": "user",
                    "content":
                    f"""
QUESTION:
{query}

CONTEXT:
{context}
"""
                }
            ]
        )
    )

    return (
        response
        .choices[0]
        .message
        .content
    )

In [69]:
for i, chunk in enumerate(parent_chunks[:3], start=1):

    print("=" * 80)
    print(f"PARENT CHUNK {i}")
    print("=" * 80)

    print(chunk.metadata)

    print("\n")

    print(chunk.page_content[:1000])

In [68]:
parent_chunks = []
seen_chunks = set()

for doc in hq_results:

    parent_id = doc.metadata["parent_chunk_id"]

    if (
        parent_id in chunk_lookup
        and parent_id not in seen_chunks
    ):

        parent_chunks.append(
            chunk_lookup[parent_id]
        )

        seen_chunks.add(parent_id)

print(
    f"Retrieved {len(parent_chunks)} parent chunks"
)

Retrieved 0 parent chunks


In [70]:
context = "\n\n".join(
    chunk.page_content
    for chunk in parent_chunks
)

response = client.chat.completions.create(
    model=model_name,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": """
Answer ONLY using the provided context.

If the answer cannot be fully supported,
say so explicitly.

Provide a concise analytical answer.
"""
        },
        {
            "role": "user",
            "content": f"""
QUESTION:
{query}

CONTEXT:
{context}
"""
        }
    ]
)

final_answer = (
    response
    .choices[0]
    .message
    .content
)

print(final_answer)

Based on the provided context (which is empty), I do not have enough information to provide a specific answer. However, I can provide a general answer.

A board member should ask about the following risks that could prevent Tesla from meeting production, delivery, or scaling expectations:

1. Supply chain disruptions: Are there any potential risks or bottlenecks in the supply chain that could impact production or delivery?
2. Manufacturing capacity constraints: Does Tesla have sufficient manufacturing capacity to meet production targets, and are there any plans to expand or upgrade existing facilities?
3. Technology and innovation risks: Are there any potential risks or challenges associated with the development and implementation of new technologies, such as autonomous driving or battery advancements?
4. Regulatory and compliance risks: Are there any regulatory or compliance risks that could impact Tesla's ability to meet production, delivery, or scaling expectations?
5. Talent acquis

In [71]:
output = {
    "question_id": "HQ1",
    "user_query": query,

    "retrieved_hypothetical_questions": [
        {
            "hypothetical_question":
                doc.page_content,

            "parent_chunk_id":
                doc.metadata["parent_chunk_id"],

            "section":
                doc.metadata["section"],

            "year":
                doc.metadata["year"]
        }

        for doc in hq_results
    ],

    "parent_chunks_used": [
        {
            "chunk_id":
                doc.metadata["chunk_id"],

            "source_doc":
                doc.metadata["source_doc"],

            "section":
                doc.metadata["section"],

            "year":
                doc.metadata["year"]
        }

        for doc in parent_chunks
    ],

    "final_answer":
        final_answer,

    "citations": [],

    "comparison_with_baseline":
        ""
}

In [53]:
hypothetical_questions[0], hypothetical_questions[1], hypothetical_questions[2]

(Document(metadata={'parent_chunk_id': '34ead2d5-bcbd-482f-8104-46662b3cd246', 'source_doc': 'tsla-10ka_20211231-gen.pdf', 'year': 2021, 'section': 'Unknown'}, page_content='What is the name of the registrant as specified in its charter?'),
 Document(metadata={'parent_chunk_id': '34ead2d5-bcbd-482f-8104-46662b3cd246', 'source_doc': 'tsla-10ka_20211231-gen.pdf', 'year': 2021, 'section': 'Unknown'}, page_content="What is the address of Tesla's principal executive offices?"),
 Document(metadata={'parent_chunk_id': '34ead2d5-bcbd-482f-8104-46662b3cd246', 'source_doc': 'tsla-10ka_20211231-gen.pdf', 'year': 2021, 'section': 'Unknown'}, page_content="What is the trading symbol of Tesla's common stock?"))

In [72]:
import json

with open(
    "HQ1.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=4,
        ensure_ascii=False
    )

print("HQ1.json saved successfully")

HQ1.json saved successfully


In [73]:
import json

def run_hq_benchmark(
    question_id,
    query
):

    # ----------------------------------
    # HQ Retrieval
    # ----------------------------------

    hq_results = hq_retriever.invoke(
        query
    )

    # ----------------------------------
    # Parent Chunk Retrieval
    # ----------------------------------

    parent_chunks = []
    seen_chunks = set()

    for doc in hq_results:

        parent_id = doc.metadata[
            "parent_chunk_id"
        ]

        if (
            parent_id in chunk_lookup
            and parent_id not in seen_chunks
        ):

            parent_chunks.append(
                chunk_lookup[parent_id]
            )

            seen_chunks.add(
                parent_id
            )

    # ----------------------------------
    # Context Creation
    # ----------------------------------

    context = "\n\n".join(
        chunk.page_content
        for chunk in parent_chunks
    )

    # ----------------------------------
    # Answer Generation
    # ----------------------------------

    response = (
        client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {
                    "role": "system",
                    "content": """
Answer ONLY using the provided context.

Do not invent facts.

If evidence is incomplete,
state that clearly.

Provide an analytical answer.
"""
                },
                {
                    "role": "user",
                    "content": f"""
QUESTION:
{query}

CONTEXT:
{context}
"""
                }
            ]
        )
    )

    final_answer = (
        response
        .choices[0]
        .message
        .content
    )

    # ----------------------------------
    # Build JSON
    # ----------------------------------

    output = {

        "question_id":
            question_id,

        "user_query":
            query,

        "retrieved_hypothetical_questions": [

            {
                "hypothetical_question":
                    doc.page_content,

                "parent_chunk_id":
                    doc.metadata[
                        "parent_chunk_id"
                    ],

                "section":
                    doc.metadata[
                        "section"
                    ],

                "year":
                    doc.metadata[
                        "year"
                    ]
            }

            for doc in hq_results
        ],

        "parent_chunks_used": [

            {
                "chunk_id":
                    doc.metadata[
                        "chunk_id"
                    ],

                "source_doc":
                    doc.metadata[
                        "source_doc"
                    ],

                "section":
                    doc.metadata[
                        "section"
                    ],

                "year":
                    doc.metadata[
                        "year"
                    ]
            }

            for doc in parent_chunks
        ],

        "final_answer":
            final_answer,

        "citations": [],

        "comparison_with_baseline":
            ""
    }

    return output

In [74]:
benchmark_questions = {

    "HQ1":
    "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",

    "HQ2":
    "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",

    "HQ3":
    "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",

    "HQ4":
    "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say cybersecurity?"
}

We can now index these hypothetical questions into a new collection.

In [75]:
all_results = {}

for qid, query in benchmark_questions.items():

    print(
        f"\nRunning {qid}..."
    )

    result = run_hq_benchmark(
        qid,
        query
    )

    all_results[qid] = result

    with open(
        f"{qid}.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            result,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(
        f"{qid}.json saved"
    )


Running HQ1...
HQ1.json saved

Running HQ2...
HQ2.json saved

Running HQ3...
HQ3.json saved

Running HQ4...
HQ4.json saved


In [76]:
with open(
    "assignment_A003_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print(
    "assignment_A003_results.json saved"
)

assignment_A003_results.json saved


In [79]:
def retrieve_baseline(query):

    docs = retriever.invoke(query)

    return docs

In [80]:
def evidence_quality(num_chunks):

    if num_chunks >= 5:
        return "High"

    elif num_chunks >= 3:
        return "Medium"

    return "Low"

In [77]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [82]:
comparison_results = []

for qid, query in benchmark_questions.items():

    print(f"\nEvaluating {qid}")

    baseline_docs = retrieve_baseline(query)

    hq_docs = hq_retriever.invoke(query)

    parent_chunks = []
    seen = set()

    for doc in hq_docs:

        parent_id = doc.metadata["parent_chunk_id"]

        if (
            parent_id in chunk_lookup
            and parent_id not in seen
        ):

            parent_chunks.append(
                chunk_lookup[parent_id]
            )

            seen.add(parent_id)

    baseline_quality = evidence_quality(
        len(baseline_docs)
    )

    improved_quality = evidence_quality(
        len(parent_chunks)
    )

    comparison_results.append({

        "Question": qid,

        "Baseline Evidence Quality":
            baseline_quality,

        "Improved Evidence Quality":
            improved_quality,

        "Improvement Observed": "",

        "Failure Mode": ""

    })


Evaluating HQ1

Evaluating HQ2

Evaluating HQ3

Evaluating HQ4


In [83]:
import pandas as pd

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,Question,Baseline Evidence Quality,Improved Evidence Quality,Improvement Observed,Failure Mode
0,HQ1,High,Low,,
1,HQ2,High,Low,,
2,HQ3,High,Low,,
3,HQ4,High,Low,,


In [84]:
for qid, query in benchmark_questions.items():

    print(f"\nEvaluating {qid}")

    # ------------------------
    # Baseline Retrieval
    # ------------------------

    baseline_docs = retrieve_baseline(
        query
    )

    # ------------------------
    # HQ Retrieval
    # ------------------------

    hq_docs = hq_retriever.invoke(
        query
    )

    # ------------------------
    # Parent Chunks
    # ------------------------

    parent_chunks = []

    seen = set()

    for doc in hq_docs:

        parent_id = doc.metadata[
            "parent_chunk_id"
        ]

        if (
            parent_id in chunk_lookup
            and parent_id not in seen
        ):

            parent_chunks.append(
                chunk_lookup[parent_id]
            )

            seen.add(parent_id)

    # ------------------------
    # Quality
    # ------------------------

    baseline_quality = (
        evidence_quality(
            len(baseline_docs)
        )
    )

    improved_quality = (
        evidence_quality(
            len(parent_chunks)
        )
    )

    comparison_results.append({

        "Question":
            qid,

        "Baseline Evidence Quality":
            baseline_quality,

        "Improved Evidence Quality":
            improved_quality,

        "Improvement Observed":
            "",

        "Failure Mode":
            ""
    })


Evaluating HQ1

Evaluating HQ2

Evaluating HQ3

Evaluating HQ4


In [85]:
import pandas as pd

comparison_df = pd.DataFrame(
    comparison_results
)

comparison_df

,Question,Baseline Evidence Quality,Improved Evidence Quality,Improvement Observed,Failure Mode
0,HQ1,High,Low,,
1,HQ2,High,Low,,
2,HQ3,High,Low,,
3,HQ4,High,Low,,
4,HQ1,High,Low,,
5,HQ2,High,Low,,
6,HQ3,High,Low,,
7,HQ4,High,Low,,


# Conclusion

This project implemented a production-style Retrieval-Augmented Generation pipeline for Tesla annual reports.

Key achievements:

- Built a semantic search system using ChromaDB
- Integrated Llama 3.1 70B for grounded generation
- Implemented advanced retrieval using hypothetical questions
- Improved retrieval quality for complex financial queries

The approach demonstrates how RAG can be applied to enterprise knowledge bases and financial document analysis.

# Future Enhancements

Potential improvements include:

1. Hybrid Search
   - BM25 + Vector Search

2. Reranking Models
   - Cross Encoder Rerankers

3. Parent-Child Retrieval

4. Multi-Query Retrieval

5. Metadata Filtering
   - Year-based filtering
   - Report section filtering

6. Automated Evaluation Framework

7. Citation Generation
   - Return source pages with answers